In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '3'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
import jax
import jax.numpy as jnp
from jax import random
import numpy as np
import optax
import time
from dataset import *
import matplotlib.pyplot as plt
from functools import partial
from tqdm import trange
from flax.training import train_state
import orbax.checkpoint as ocp
from jax import tree_util

import logging
logging.getLogger("jax").setLevel(logging.ERROR) 

seed = 1234
key = jax.random.PRNGKey(seed)
np.random.seed(seed)
data_config = DatasetConfig()

DEFAULT_DTYPE = data_config.dtype


@jax.jit
def mse(y_pre, y_true):
    assert y_pre.shape == y_true.shape, f"Shape mismatch: y_pre.shape = {y_pre.shape}, y_true.shape = {y_true.shape}"
    return jnp.mean(jnp.square(y_pre - y_true))


@jax.jit
def improved_mse(y_pre, y_true):
    assert y_pre.shape == y_true.shape, f"Shape mismatch: y_pre.shape = {y_pre.shape}, y_true.shape = {y_true.shape}"
    return jnp.mean((jnp.square(y_pre - y_true) + 1.0e-12)**(1/3))


def l2_relative_error(y_pre, y_true, dim=(1,2)):
    assert y_pre.shape == y_true.shape, f"Shape mismatch: y_pre.shape = {y_pre.shape}, y_true.shape = {y_true.shape}"

    return jnp.linalg.norm(y_pre-y_true, ord=2, axis=dim) / jnp.linalg.norm(y_true, ord=2, axis=dim)


@partial(jax.jit, static_argnames='state_forward')
def grad2_basis_f(x, params, state_forward):
    def f(x_single):
        x_single = x_single[None, :]
        output = state_forward(params, x_single)
        return output[0]

    def df_dxx(x):
        return jax.jacfwd(jax.jacfwd(f))(x)
    
    return jnp.squeeze(jax.vmap(lambda xi: df_dxx(xi))(x))


@jax.jit
def numerical_second_derivative_5point(f, x):
    h = x[1] - x[0]
    ddf = jnp.zeros_like(f)

    ddf = ddf.at[:, 2:-2].set(
        (-f[:, 4:] + 16 * f[:, 3:-1] - 30 * f[:, 2:-2] + 16 * f[:, 1:-3] - f[:, :-4]) / (12 * h ** 2)
    )

    ddf = ddf.at[:, 0].set((2 * f[:, 0] - 5 * f[:, 1] + 4 * f[:, 2] - f[:, 3]) / (h ** 2))
    ddf = ddf.at[:, 1].set((f[:, 0] - 2 * f[:, 1] + f[:, 2]) / (h ** 2))
    ddf = ddf.at[:, -2].set((f[:, -3] - 2 * f[:, -2] + f[:, -1]) / (h ** 2))
    ddf = ddf.at[:, -1].set((2 * f[:, -1] - 5 * f[:, -2] + 4 * f[:, -3] - f[:, -4]) / (h ** 2))

    return ddf


def save_state(state, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    checkpointer = ocp.PyTreeCheckpointer()
    checkpointer.save(path, state, force=True)


def restore_state(path, dummy_state):
    checkpointer = ocp.PyTreeCheckpointer()
    state = checkpointer.restore(path, item=dummy_state)
    return state

    
def create_train_state(model, rng, example_input, lr, eta_min, betas=(0.9, 0.999), epochs=None):
    if isinstance(example_input, (tuple, dict)):
        params = model.init(rng, *example_input) if isinstance(example_input, tuple) else model.init(rng, **example_input)
    else:
        params = model.init(rng, example_input)

    if epochs is not None:
        scheduler = optax.cosine_decay_schedule(init_value=lr,
                                                decay_steps=epochs,
                                                alpha=eta_min/lr)
    else:
        scheduler = lr

    tx = optax.adamw(
        learning_rate=scheduler if epochs is not None else lr,
        b1=betas[0],
        b2=betas[1],
        eps=1e-8,
        weight_decay=1e-5
    )

    # 返回 TrainState
    return train_state.TrainState.create(
        apply_fn=model.apply,
        params=params,
        tx=tx
    )


@partial(jax.jit, static_argnames='bnn')
def sample_params(sigmas_w, sigmas_b, keys, bnn):
    def single_sample(sigma_w, sigma_b, key):
        return bnn.hyper_initial(sigma_w, sigma_b, key)

    # vmap over sigma and key
    params_all = jax.vmap(single_sample)(sigmas_w, sigmas_b, keys)

    params_concat = tree_util.tree_map(lambda x: x.reshape((-1,) + x.shape[2:]), params_all)
    return params_concat

In [ ]:
seed = 1234
key = jax.random.PRNGKey(seed)
# sample bnn
sigma = 16
x = data_config.x
bnn_layer_size = [1, 100, 1]

bnn = BNN_sample(bnn_layer_size, 10000)
key, subkey = jax.random.split(key)
params = bnn.hyper_initial(sigma, sigma, key)

samples_train, d2u_dx2 = batch_compute_manual_derivatives_xt(x, params, 0)

cov_matrix = np.cov(samples_train, rowvar=False)

# samples_train: shape [num_samples, num_points]
X = samples_train                   # shape [num_samples, num_points]
N = X.shape[0]
cov_manual = (X.T @ X) / (N - 1)  # shape [num_points, num_points]

plt.rcdefaults()
plt.figure()
plt.plot(x, samples_train[:10, :].T)
plt.show()

plt.figure(figsize=(8, 6))
plt.imshow(cov_manual, cmap='viridis')
plt.colorbar(label='Covariance')
plt.title("Covariance Matrix of BNN Samples (Points vs. Points)")
plt.xlabel("Spatial Point Index")
plt.ylabel("Spatial Point Index")
plt.tight_layout()
plt.show()

numerical_d2u_dx2 = numerical_second_derivative_5point(samples_train, x.squeeze())
plt.figure()
plt.plot(x, numerical_d2u_dx2[:10, :].T, linestyle='-')
plt.plot(x, d2u_dx2[:10, :].T, linestyle='--')
plt.show()

f = f_func(samples_train, d2u_dx2)
plt.figure()
plt.plot(f[:10, :].T)
plt.show()

In [ ]:
DeepONet_config = DeepONetConfig()
xi_hidden_dim = DeepONet_config.branch.hidden_neuron
xi_output_dim = DeepONet_config.branch.output_neuron
xi_hidden_layers = DeepONet_config.branch.hidden_layers

mlp_xi = ResMLP(xi_hidden_dim, xi_output_dim, xi_hidden_layers, DeepONet_config.branch.activation_fn)
mlp_basis = TrunkNet(256, data_config.basis_fn_dim, 4, jax.nn.tanh, 14.0)

key, subkey = random.split(key)
xi_init = jnp.zeros((1, data_config.x_num))
trunknet_init = jnp.zeros((1, 1))

iterations = 300000
lr = 1.0e-4
eta_min_lr = 1.0e-7


key, key1, key2 = random.split(key, 3)
state_xi_pretrain = create_train_state(mlp_xi, key1, xi_init, lr, eta_min_lr, epochs=iterations)
state_basis = create_train_state(mlp_basis, key2, trunknet_init, lr, eta_min_lr, epochs=iterations)

In [ ]:
bnn_layer_size = [1, 100, 1]
bnn = BNN_sample(bnn_layer_size, 100)
x = data_config.x 

In [ ]:
@jax.jit
def train_step(state_xi, state_basis, u_input, f_input, trunk_input):
    def loss_fn(params_xi, params_basis):        
        phi_basis = state_basis.apply_fn(params_basis, trunk_input)
        grad2_phi_basis = grad2_basis_f(trunk_input, params_basis, state_basis.apply_fn)
        
        phi_xi_u = state_xi.apply_fn(params_xi, u_input)
        u_pre = jnp.einsum('ik,jk->ij', phi_xi_u, phi_basis)
        loss1 = mse(u_pre, u_input)
        
        phi_xi_f = state_xi.apply_fn(params_xi, f_input)
        f_pre1 = jnp.einsum('ik,jk->ij', phi_xi_f, phi_basis)
        loss2 = mse(f_pre1, f_input)

        u_pre_d2x = jnp.einsum('ik,jk->ij', phi_xi_u, grad2_phi_basis)
        f_pre2 = f_func(u_pre, u_pre_d2x)
        loss3 = mse(f_pre2, f_input)

        loss = 100 * loss1 + 10 * loss2 + loss3

        return loss, (loss1, loss2, loss3)
    (loss, (loss1, loss2, loss3)), grads = jax.value_and_grad(loss_fn, argnums=(0,1), has_aux=True)(state_xi.params, state_basis.params)
    grad_xi, grad_basis = grads

    state_xi = state_xi.apply_gradients(grads=grad_xi)
    state_basis = state_basis.apply_gradients(grads=grad_basis)

    return state_xi, state_basis, loss, loss1, loss2, loss3

sigma_list_w = data_config.sigma_list
sigma_list_b = sigma_list_w
start_time = time.time()
for i in range(iterations):
    keys = random.split(key, len(sigma_list_w)+1)
    key, subkey = keys[0], keys[1:]
    params_all_concat = sample_params(sigma_list_w, sigma_list_b, subkey, bnn)

    u_sin_in, d2u_dx2_sin_in = batch_compute_manual_derivatives_xt(x, params_all_concat, 0)
    f_sin_in = f_func(u_sin_in, d2u_dx2_sin_in)

    u_input = u_sin_in
    f_input = f_sin_in

    state_xi_pretrain, state_basis, loss, loss1, loss2, loss3 = train_step(state_xi_pretrain, state_basis, u_input, f_input, x)

    if (i+1) % 1000 == 0:
        print(f'Step: [{i}/{iterations}], loss: {loss:.3e}, loss1:{loss1:.3e}, loss2:{loss2:.3e}, loss3:{loss3:.3e}')

end_time = time.time()
elapsed_time = end_time - start_time
print(f'Duration:{elapsed_time:.2f}s')

In [ ]:
state_basis_forward = jax.jit(state_basis.apply_fn)

In [ ]:
basis = state_basis_forward(state_basis.params, x)
grad2_basis = grad2_basis_f(x, state_basis.params, state_basis_forward)

rank = np.linalg.matrix_rank(basis)
print(f"矩阵秩: {rank} / {data_config.basis_fn_dim}")

cond = np.linalg.cond(basis)
print(f"条件数: {cond:.3e}")


plt.figure()
plt.plot(x, basis)
plt.title('basis')
plt.show()

plt.figure()
plt.plot(x, grad2_basis)
plt.title('grad2_basis')
plt.show()

In [ ]:
state_xi_pretrain_forward = jax.jit(state_xi_pretrain.apply_fn)

In [ ]:
# test
bnn = BNN_sample(bnn_layer_size, 500)
keys = random.split(key, len(sigma_list_w)+1)
key, subkey = keys[0], keys[1:]
params_all_concat = sample_params(sigma_list_w, sigma_list_b, subkey, bnn)

u_sin_in, d2u_dx2_sin_in = batch_compute_manual_derivatives_xt(x, params_all_concat, 0)
f_sin_in = f_func(u_sin_in, d2u_dx2_sin_in)

u_input = u_sin_in
f_input = f_sin_in

u_xi_test = state_xi_pretrain_forward(state_xi_pretrain.params, u_input)
u_pre = jnp.einsum('ik,jk->ij', u_xi_test, basis)

f_xi_test = state_xi_pretrain_forward(state_xi_pretrain.params, f_input)
f_pre = jnp.einsum('ik,jk->ij', f_xi_test, basis)

for i, sigma in enumerate(sigma_list_w):
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(u_input[i*500:i*500+10, :].T, linestyle='-')
    plt.plot(u_pre[i*500:i*500+10, :].T, linestyle='--')
    plt.title(f"sigma={int(sigma)}, u")

    plt.subplot(1,2,2)
    plt.plot(f_input[i*500:i*500+10, :].T, linestyle='-')
    plt.plot(f_pre[i*500:i*500+10, :].T, linestyle='--')
    plt.title(f"sigma={int(sigma)}, f")
    plt.tight_layout()
    plt.show()


l2_error_u = l2_relative_error(u_pre, u_input, dim=(1))
l2_error_u_mean = jnp.mean(l2_error_u)
l2_error_u_std = jnp.std(l2_error_u)
print(f'l2_error_u mean={l2_error_u_mean:.3e}, l2_error_u std={l2_error_u_std:.3e}')

l2_error_f = l2_relative_error(f_pre, f_input, dim=(1))
l2_error_f_mean = jnp.mean(l2_error_f)
l2_error_f_std = jnp.std(l2_error_f)
print(f'l2_error_f mean={l2_error_f_mean:.3e}, l2_error_f std={l2_error_f_std:.3e}')

In [ ]:
save_state(state_xi_pretrain, os.path.join(data_config.checkpoint_dir, "deeponet_state_xi"))
save_state(state_basis, os.path.join(data_config.checkpoint_dir, "deeponet_state_basis"))